# Tool Configuration - Control Invocation Behavior

## Purpose
Learn how to control when and how agents use tools through configuration. This enables patterns like forcing specific tool usage, preventing tool calls, or customizing execution flow for different use cases.

## Key Concepts
- **tool_choice**: Controls which tools can be invoked (auto/required/none/specific)
- **tool_use_behavior**: Defines execution flow after tool calls
- **ModelSettings**: Configuration object for tool behavior
- **Deterministic Behavior**: Force predictable tool usage patterns

## Installation

In [ ]:
#!pip install openai
#!pip install openai-agents
#!pip install aws-bedrock-token-generator

## Authentication Setup

In [ ]:
model_id = "openai.gpt-5.5"

In [ ]:
from openai import AsyncOpenAI
from agents import (
    set_default_openai_client,
    set_default_openai_api,
    set_tracing_disabled,
)
from aws_bedrock_token_generator import provide_token

client = AsyncOpenAI(
    api_key=provide_token(),
    base_url="https://bedrock-mantle.us-east-1.api.aws/openai/v1",
    project="default"
)

set_default_openai_client(client)
set_default_openai_api("responses")
set_tracing_disabled(True)  # OpenAI-platform tracing can't reach Mantle

## Import Libraries

Import `ModelSettings` for tool configuration:

In [ ]:
import asyncio

from agents import Agent, Runner, function_tool, ModelSettings

## Part 1: Tool Choice - Force Specific Tool Usage

Use `ModelSettings(tool_choice=...)` to control which tools the agent can invoke.

**Available Options**:
- `"auto"`: Agent decides which tool to use based on context (default)
- `"required"`: Agent must use at least one tool (cannot answer without tools)
- `"none"`: Agent cannot use any tools (text response only)
- `"tool_name"`: Agent must use the specified tool

💡 **Use Case**: Force specific tool when you know exactly what needs to happen.

### Step 1: Define Tool

In [ ]:
@function_tool
def get_weather(city: str) -> str:
    """Returns weather info for the specified city."""
    return f"The weather in {city} is sunny"

### Step 2: Force Specific Tool

Set `tool_choice="get_weather"` to guarantee this tool is called:

🎯 **Result**: Agent will always call `get_weather`, even if the query is ambiguous!

In [ ]:
agent = Agent(
    model=model_id,
    name="Weather Agent",
    instructions="Retrieve weather details.",
    tools=[get_weather],
    model_settings=ModelSettings(tool_choice="get_weather")  # Force this tool
)

In [ ]:
result = await Runner.run(agent, "How is the weather in Leuven?")
print(result.final_output)

## Part 2: Tool Use Behavior - Control Execution Flow

Use `tool_use_behavior` to define what happens after a tool is called.

**Available Options**:
- `"run_llm_again"`: After tool execution, run the LLM again to process results (default)
- `"stop_on_first_tool"`: Stop execution immediately after first tool call
- Custom function: Provide your own logic to determine behavior

⚡ **Key Difference**:
- `run_llm_again`: Tool result → LLM processes → Natural language response
- `stop_on_first_tool`: Tool result → Return immediately (raw tool output)

💡 **Use Case**: Use `stop_on_first_tool` when you just need the tool's raw output, not a synthesized answer.

### Step 3: Stop After First Tool

Configure agent to return tool result immediately without further processing:

In [ ]:
agent_stop_first = Agent(
    model=model_id,
    name="Weather Agent",
    instructions="Retrieve weather details.",
    tools=[get_weather],
    tool_use_behavior="stop_on_first_tool"  # Stop after first tool call
)

### Step 4: Run and Compare

🔍 **Watch**: Output is the raw tool result, not a natural language response!

In [ ]:
result = await Runner.run(agent_stop_first, "How is the weather in Leuven?")
print(result.final_output)

## 🎉 Congratulations!

You've completed the **Tool Configuration** notebook!